![JohnSnowLabs](https://nlp.johnsnowlabs.com/assets/images/logo.png)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JohnSnowLabs/visual-nlp-workshop/blob/master/jupyter/Dicom/SparkOCRBulkDeid.ipynb)

# DICOM BULK De-Identification Pipeline (Visual NLP)

> ⚠️ **Runs on Databricks only.** 

**What this notebook does:** takes medical images in DICOM format (`.dcm`) and removes private patient information (PHI) from them, in two places:

1. **Burned-in text** - names, dates or IDs printed *on the image pixels* are blacked out.
2. **Metadata (DICOM tags)** - the hidden header fields (patient name, birth date, ...) are cleaned.

**The workflow in plain steps**

| # | Step | What happens |
|---|------|--------------|
| 1 | Read DICOM files | Load the files from a folder |
| 2 | OCR | Read the text on the image (standard OCR *or* a vision-language model) |
| 3 | NLP | Find the sensitive words (names, dates, IDs, contacts) in that text |
| 4 | Coordinates | Work out where each sensitive word is on the image |
| 5 | De-identify | Black out those areas and clean the metadata tags |
| 6 | Check | Compare metadata before / after and look at the images |

## Cluster Configuration (Databricks)

Create a Databricks cluster with these settings (**Compute → your cluster → Advanced options → Spark config**), and restart it after any change.

| Setting | CPU cluster | GPU cluster |
|---|---|---|
| Workers | `i4i.2xlarge` × 4 (fixed, min 4 / max 4) | `g5.2xlarge` (A10G) × 4 (fixed, min 4 / max 4) |
| Driver | `i4i.2xlarge` | `g5.2xlarge` (A10G) |
| Photon acceleration | Disabled | Disabled |
| `spark.sql.legacy.allowUntypedScalaUDF` | `true` | `true` |
| `spark.executor.memoryOverhead` | `16g` | `16g` |
| `spark.task.cpus` | `4` | default |

**CPU concurrency:** each `i4i.2xlarge` has 8 vCPUs. With `spark.task.cpus = 4`, each executor runs 8 / 4 = **2 tasks at a time**, so the cluster runs at most 4 × 2 = **8 concurrent tasks**.

**Tip:** Spark is *lazy* - many cells return instantly because they only build a plan. The real work happens when data is shown, counted or written.

## Import transformers and annotators

In [0]:
import os
import json 
import sys
import shutil 

from sparknlp.annotator import *
from sparknlp.base import *
import sparknlp_jsl
from sparknlp_jsl.annotator import *

import sparkocr
from sparkocr.transformers import *
from sparkocr.utils import *
from sparkocr.enums import *
from sparkocr.schemas import BinarySchema

from pyspark.ml import PipelineModel, Pipeline
import pyspark.sql.functions as F

print(f"Spark NLP version: {sparknlp.version()}")
print(f"Spark NLP internal version: {sparknlp_jsl.version()}")
print(f"Spark OCR version: {sparkocr.version()}")

Spark NLP version: 6.4.2
Spark NLP internal version: 6.4.1
Spark OCR version: 6.4.3rc10


## Define the pipelines
The next four cells only **define** functions that build pipelines; nothing runs until they are called further down.

- **NLP pipeline** - finds sensitive text and its coordinates
- **Standard OCR pipeline** - detects text regions, then reads them
- **VLM OCR pipeline** - a vision-language model reads text and coordinates [ Requires a GPU ]
- **Finalizer** - blacks out the text and cleans the metadata

In [0]:
def nlp_deid_pipeline():
    
    document_assembler = DocumentAssembler() \
        .setInputCol("text") \
        .setOutputCol("document")

    sentence_detector = SentenceDetector() \
        .setInputCols(["document"]) \
        .setOutputCol("sentence")
    
    labels = ["NAME", "DATE", "ID", "CONTACT"]
    zeroshot_ner_deid_generic_nonMedical_medium = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_generic_nonMedical_medium", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_generic_nonMedical_medium")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)

    labels = ["IDNUM","MEDICALRECORD", "NAME","PATIENT", "PHONE"]
    zeroshot_ner_deid_subentity_nonMedical_large = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_subentity_nonMedical_large", "en", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_subentity_nonMedical_large")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)

    labels = ['CONTACT', 'DATE', 'ID', 'NAME'] 
    zeroshot_ner_deid_generic_multi_large = PretrainedZeroShotNERChunker().pretrained("zeroshot_ner_deid_generic_multi_large", "xx", "clinical/models")\
        .setInputCols("document")\
        .setOutputCol("ner_zeroshot_ner_deid_generic_multi_large")\
        .setPredictionThreshold(0.5)\
        .setLabels(labels)
    
    date_regex_matcher = RegexMatcherInternalModel.pretrained("date_matcher","en","clinical/models") \
        .setInputCols(["sentence"]) \
        .setOutputCol("date_chunk")
    
    chunk_merger = ChunkMergeApproach()\
        .setInputCols("ner_zeroshot_ner_deid_generic_nonMedical_medium", "ner_zeroshot_ner_deid_subentity_nonMedical_large","ner_zeroshot_ner_deid_generic_multi_large", "date_chunk")\
        .setOutputCol("merged_chunk")\
        .setMergeOverlapping(True)

    position_finder = PositionFinder() \
        .setInputCols("merged_chunk") \
        .setOutputCol("coordinates") \
        .setSplitOnNewLines(True) \
        .setPageMatrixCol("positions") \
        .setIgnoreSchema(True)

    nlp_pipeline = Pipeline(stages=[
        document_assembler,
        sentence_detector,
        zeroshot_ner_deid_generic_nonMedical_medium,
        zeroshot_ner_deid_subentity_nonMedical_large,
        zeroshot_ner_deid_generic_multi_large,
        date_regex_matcher,
        chunk_merger,
        position_finder
    ])

    empty_data = spark.createDataFrame([[""]]).toDF("text")
    nlp_model = nlp_pipeline.fit(empty_data)
    return nlp_model

In [0]:
def ocr_pipeline_builder(scale=2.0, frame_limit=5, compression_quality=80, use_gpu=False):
    """
    ImageTextDetector can run on CPU/GPU
    ImageToTextV3 can only run on CPU
    """
    
    dicom_to_image = DicomToImageV3() \
        .setInputCols(["content"]) \
        .setOutputCol("image") \
        .setCompressionMode("enabled") \
        .setKeepInput(False) \
        .setMemoryOptimized(True) \
        .setCompressionQuality(compression_quality) \
        .setScale(scale) \
        .setFrameLimit(frame_limit) \
        .setFrameSamplingStrategy(FrameSamplingStrategy.CONSECUTIVE) \
        .setCompressImage(True) \
        .setFrameDimsCol("frame_dims")
    
    text_detector = ImageTextDetector.pretrained("image_text_detector_mem_opt", "en", "clinical/ocr") \
        .setInputCol("image") \
        .setOutputCol("text_regions") \
        .setWithRefiner(True) \
        .setUseGPU(use_gpu) \
        .setWidth(0)
    
    ocr = ImageToTextV3() \
        .setInputCols(["image", "text_regions"]) \
        .setOutputCol("text")
    
    ocr_pipeline = PipelineModel(stages=[
        dicom_to_image,
        text_detector,
        ocr
    ])

    return ocr_pipeline

In [0]:
def ocr_vlm_pipeline_builder(scale=2.0, frame_limit=5, compression_quality=80):
    
    dicom_to_image = DicomToImageV3() \
        .setInputCols(["content"]) \
        .setOutputCol("image") \
        .setCompressionMode("enabled") \
        .setKeepInput(False) \
        .setMemoryOptimized(True) \
        .setCompressionQuality(compression_quality) \
        .setScale(scale) \
        .setFrameLimit(frame_limit) \
        .setFrameSamplingStrategy(FrameSamplingStrategy.CONSECUTIVE) \
        .setCompressImage(True) \
        .setFrameDimsCol("frame_dims")
    
    caption_assembler = DocumentAssembler() \
        .setInputCol("caption") \
        .setOutputCol("caption_document")

    schema_converter_assembler = ImageSchemaConverter() \
        .setInputCol("image") \
        .setOutputCol("image_assembler") \
        .setOutputSchema(ImageSchemaConversion.ASSEMBLER) \
        .setKeepInput(False)

    ocr = MedicalVisionLLM.pretrained("jsl-ocr-gguf-vlm1", "en", "clinical/ocr") \
        .setInputCols(["caption_document", "image_assembler"]) \
        .setOutputCol("completions") \
        .setNGpuLayers(99) \
        .setNCtx(32768) \
        .setNParallel(1) \
        .setNBatch(2048) \
        .setNUbatch(1024) \
        .setNPredict(1024) \
        .setTemperature(0.01) \
        .setTopK(1) \
        .setTopP(1.0) \
        .setRepeatPenalty(1.03) \
        .setRepeatLastN(256) \
        .setStopStrings(["<\uff5chy_Assistant\uff5c>", "<\uff5chy_place\u2581holder\u2581no\u25812\uff5c>"]) \
        .setMinKeep(0) \
        .setNProbs(0) \
        .setBatchSize(1) \
        .setDisableLog(False)

    coordinate_extract = DocumentCoordinatesToText() \
        .setInputCol("completions") \
        .setImageDimsCol("frame_dims") \
        .setOutputCol("text") \
        .setPageMatrixCol("positions") \
        .setRegionCol("regions")
    
    vlm_ocr_pipeline = PipelineModel(stages=[
        dicom_to_image,
        caption_assembler,
        schema_converter_assembler,
        ocr,
        coordinate_extract
    ])

    return vlm_ocr_pipeline

In [0]:
def dicom_finalizer_stages(scale, partition_size=None, strategy_file_path=None):

    assert partition_size != None and isinstance(partition_size, int), "Partition Size Cannot be None and should be Integer."

    draw_regions = DicomDrawRegions() \
        .setInputCol("path") \
        .setInputRegionsCol("coordinates") \
        .setOutputCol("dicom_pixel_cleaned") \
        .setAggCols(["path"]) \
        .setKeepInput(True) \
        .setIgnoreSchema(True) \
        .setOutputPartitionSize(partition_size) \
        .setScaleFactor(1 / scale)

    if strategy_file_path is not None:

        print(f"Strategy File Provided : {strategy_file_path}")

        dicom_deidentifier = DicomMetadataDeidentifier() \
            .setInputCols(["dicom_pixel_cleaned"]) \
            .setOutputCol("dicom_final_cleaned") \
            .setStrategyFile(strategy_file_path) \
            .setRemovePrivateTags(True)
    else:

        print("Strategy File Not Provided : Using Default")

        dicom_deidentifier = DicomMetadataDeidentifier() \
            .setInputCols(["dicom_pixel_cleaned"]) \
            .setOutputCol("dicom_final_cleaned") \
            .setRemovePrivateTags(True)

    original_metadata = DicomToMetadata() \
        .setInputCol("path") \
        .setOutputCol("metadata_original") \
        .setKeepInput(True) \
        .setExtractTagForNer(False)

    deid_metadata = DicomToMetadata() \
        .setInputCol("dicom_final_cleaned") \
        .setOutputCol("metadata_final") \
        .setKeepInput(True) \
        .setExtractTagForNer(False)

    dicom_pipeline = PipelineModel(stages=[
        draw_regions,
        dicom_deidentifier,
        original_metadata,
        deid_metadata
    ])

    return dicom_pipeline

## Settings
Change the values in the next cell to fit your data. A summary printed afterwards lets you double-check them.

**Settings. Edit the values in this cell to match your data; everything else reads from here**

- scale: image zoom used for OCR. Higher = better reading of small text but slower and more memory.
- frame_limit: max number of frames processed per DICOM file (multi-frame files can be huge).
- compression_quality: 1-100. Higher = better image quality but bigger size.
- useGPU: True if the cluster has a GPU. Set to False for CPU-only clusters.

In [0]:
# DICOM IMAGE SETTING
scale = 2.0
frame_limit = 5
compression_quality = 80

#GPU SETTING
useGPU = True 

# I/O SETTING
dicom_load_path = "/Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/input"
root_save_path = "/Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/output"

ocr_save_path = os.path.join(root_save_path, "ocr")

nlp_save_path = os.path.join(root_save_path, "nlp")

In [0]:
# ---------------------------------------------------------------
# CONFIG SUMMARY
# ---------------------------------------------------------------
def _section(title, rows, width=22):
    print(f"\n  {title}")
    print("  " + "─" * 70)
    for key, value in rows:
        print(f"  {key:<{width}} {value}")

print("═" * 74)
print("  DICOM DE-IDENTIFICATION PIPELINE · CONFIGURATION")
print("═" * 74)

_section("🖼  DICOM IMAGE SETTINGS", [
    ("scale", scale),
    ("frame_limit", frame_limit),
    ("compression_quality", compression_quality),
])

_section("⚡ GPU SETTINGS", [
    ("useGPU", f"{'ENABLED ✅' if useGPU else 'DISABLED ❌ (CPU only)'}"),
])

_section("📂 INPUT", [
    ("dicom_load_path", dicom_load_path),
])

_section("💾 OUTPUT", [
    ("root_save_path", root_save_path),
    ("ocr_save_path", ocr_save_path),
    ("nlp_save_path", nlp_save_path),
], width=26)

print("\n" + "═" * 74)

══════════════════════════════════════════════════════════════════════════
  DICOM DE-IDENTIFICATION PIPELINE · CONFIGURATION
══════════════════════════════════════════════════════════════════════════

  🖼  DICOM IMAGE SETTINGS
  ──────────────────────────────────────────────────────────────────────
  scale                  2.0
  frame_limit            5
  compression_quality    80

  ⚡ GPU SETTINGS
  ──────────────────────────────────────────────────────────────────────
  useGPU                 ENABLED ✅

  📂 INPUT
  ──────────────────────────────────────────────────────────────────────
  dicom_load_path        /Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/input

  💾 OUTPUT
  ──────────────────────────────────────────────────────────────────────
  root_save_path             /Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/output
  ocr_save_path              /Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100

## Define NLP De-Identification Pipeline

In [0]:
ner_coordinate_pipeline = nlp_deid_pipeline()

zeroshot_ner_deid_generic_nonMedical_medium download started this may take some time.
Approximate size to download 753.2 MB
[OK!]
zeroshot_ner_deid_subentity_nonMedical_large download started this may take some time.
Approximate size to download 1.7 GB
[OK!]
zeroshot_ner_deid_generic_multi_large download started this may take some time.
Approximate size to download 1.5 GB
[OK!]
date_matcher download started this may take some time.
Approximate size to download 2.4 KB
[OK!]
🏃 View run funny-boar-850 at: https://dbc-3d4c44aa-a512.cloud.databricks.com/ml/experiments/2163745168702511/runs/3fa89f5a006741fa96d3c802c8f15e28
🧪 View experiment at: https://dbc-3d4c44aa-a512.cloud.databricks.com/ml/experiments/2163745168702511


## Choose ONE OCR Method

> **Run either Option A or Option B, not both.** They share the names `ocr_model` and `ocr_result`, so the second one overwrites the first.

| | **Option A: Standard OCR** | **Option B: VLM OCR** |
|---|---|---|
| How it works | Detects text regions, then reads them | One vision-language model reads text and coordinates |
| GPU | Optional | Required |

Run all cells of your chosen section, skip the other, then continue with **Dump OCR Results To Disk**.

Speed Metrics:

| Pipeline | Text detection | OCR | Device | Runtime |
|---|---|---|---|---|
| ImageTextDetector + ImageToTextV3 | ImageTextDetector (CPU) | ImageToTextV3 (CPU) | CPU only | 50 mins |
| ImageTextDetector + ImageToTextV3 | ImageTextDetector (GPU) | ImageToTextV3 (CPU) | Detector on GPU | 9 mins |
| MedicalVisionLLM | n/a | MedicalVisionLLM (GPU) | GPU | 16 mins |

### Standard OCR For Text Extraction

In [0]:
ocr_model = ocr_pipeline_builder(scale=scale, 
    frame_limit=frame_limit, 
    compression_quality=compression_quality, 
    use_gpu=useGPU)

In [0]:
# Load DICOM FILES

df = spark.read.format("binaryFile").load(dicom_load_path).withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))

print(f"Total Files in Batch: {df.count()}")

Total Files in Batch: 110


In [0]:
ocr_result = ocr_model.transform(df).drop("image")
ocr_result.columns

### Using VLM For Text Extraction

In [0]:
ocr_model = ocr_vlm_pipeline_builder(scale=scale, 
    frame_limit=frame_limit, 
    compression_quality=compression_quality)

jsl-ocr-gguf-vlm1 download started this may take some time.
Approximate size to download 1.5 GB


In [0]:
# Load DICOM FILES

vision_prompt = "Detect and recognize text in the image, and output the text coordinates in a formatted manner."

df = spark.read.format("binaryFile").load(dicom_load_path) \
    .withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", "")) \
    .withColumn("caption", F.lit(vision_prompt))
    
print(f"Total Files in Batch: {df.count()}")

Total Files in Batch: 110


In [0]:
ocr_result = ocr_model.transform(df).drop("image_assembler")
ocr_result.columns

/databricks/spark/python/pyspark/sql/dataframe.py:181: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


['exception',
 'pagenum',
 'frame_dims',
 'path',
 'modificationTime',
 'length',
 'caption',
 'caption_document',
 'completions',
 'text',
 'regions',
 'positions']

## Dump OCR Results To Disk

In [0]:
ocr_result.write.format("parquet").mode("overwrite").save(ocr_save_path)

## Generate NLP Result

In [0]:
ocr_result_df_from_disk = spark.read.format("parquet").load(ocr_save_path)

ocr_result_df_from_disk.select("exception").distinct().show(100, False)

+---------+
|exception|
+---------+
|         |
+---------+



In [0]:
nlp_result = ner_coordinate_pipeline.transform(ocr_result_df_from_disk)
nlp_result.columns

['exception',
 'pagenum',
 'frame_dims',
 'path',
 'modificationTime',
 'length',
 'caption',
 'caption_document',
 'completions',
 'text',
 'regions',
 'positions',
 'document',
 'sentence',
 'ner_zeroshot_ner_deid_generic_nonMedical_medium',
 'ner_zeroshot_ner_deid_subentity_nonMedical_large',
 'ner_zeroshot_ner_deid_generic_multi_large',
 'date_chunk',
 'merged_chunk',
 'coordinates']

In [0]:
nlp_result.write.format("parquet").mode("overwrite").save(nlp_save_path)

## Generate De-Id Result

In [0]:
nlp_result_df_from_disk = spark.read.format("parquet").load(nlp_save_path)

nlp_result_df_from_disk.select("exception").distinct().show(100, False)

+---------+
|exception|
+---------+
|         |
+---------+



In [0]:
partition_size = nlp_result_df_from_disk.rdd.getNumPartitions()

print(f"Partition Size: {partition_size}")

Partition Size: 28


In [0]:
dicom_pipeline = dicom_finalizer_stages(scale=scale, partition_size=partition_size)

Strategy File Not Provided : Using Default


In [0]:
final_result = dicom_pipeline.transform(nlp_result_df_from_disk)
final_result.columns

['exception',
 'dicom_final_cleaned',
 'path',
 'metadata_original',
 'metadata_final']

## Write De-Identified DICOMs and Metadata

This cell saves the final results of the pipeline in one pass:

1. **Cleaned DICOM files** are written to `<root_save_path>/dicom/`, parallely by each worker `.dcm` file per input, using the original file name.
2. **A metadata table** is written as Parquet to `<root_save_path>/metadata/`. It has four columns:
   - `path`: original file path
   - `metadata_original`: DICOM tags before de-identification (JSON)
   - `metadata_final`: DICOM tags after de-identification (JSON)
   - `exception`: error message if the file failed, otherwise null

In [0]:
import os
import pyarrow as pa
from pyspark.sql.types import StructType

dicom_bytes = "dicom_final_cleaned"
original_metadata_col = "metadata_original"
deid_metadata_col = "metadata_final"
exception_col = "exception"
path_col = "path"

meta_cols = [path_col, original_metadata_col, deid_metadata_col, exception_col]
out_schema = StructType([final_result.schema[c] for c in meta_cols])

# Path for Output Dicom Bytes
dicom_root = os.path.join(root_save_path, "dicom")
os.makedirs(dicom_root, exist_ok=True)

# Path for Output Dicom Metadata/Exception
parquet_path = os.path.join(root_save_path, "metadata")
os.makedirs(parquet_path, exist_ok=True)

def write_dicoms(batches):
    import os
    import pyarrow as pa
    for batch in batches:
        tbl = pa.Table.from_batches([batch])
        for p, b in zip(tbl.column("path").to_pylist(), tbl.column(dicom_bytes).to_pylist()):
            with open(os.path.join(dicom_root, os.path.basename(p)), "wb") as f:
                f.write(b)
        yield from tbl.select(meta_cols).to_batches()

# Using overwrite for this example. Change the write mode to append if running in batches,
# so that previous results are not overridden (or write each batch to a different folder).
final_result.select(path_col, dicom_bytes, original_metadata_col, deid_metadata_col, exception_col) \
    .mapInArrow(write_dicoms, out_schema) \
    .write.mode("overwrite") \
    .parquet(parquet_path)

## Check Result

In [0]:
df = spark.read.format("parquet").load(parquet_path)

df.select("path").distinct().show(10, False)

+----------------------------------------------------------------------------------------------------------------+
|path                                                                                                            |
+----------------------------------------------------------------------------------------------------------------+
|/Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/input/Synthetic_Pixel_003_copy_7.dcm |
|/Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/input/Synthetic_Pixel_010_copy_7.dcm |
|/Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/input/Synthetic_Pixel_004_copy_6.dcm |
|/Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/input/Synthetic_Pixel_007_copy_10.dcm|
|/Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/input/Synthetic_Pixel_009_copy_1.dcm |
|/Volumes/visual_nlp/default/visual_nlp/dicoms/synthetic_dicoms/bloated_100/inpu

In [0]:
print(f'Total De-Identified Files : {df.select("path").distinct().count()}')

Total De-Identified Files : 110


In [0]:
metadata_result = df.filter(F.col("path").contains("Synthetic_Pixel_007_copy_4.dcm")).collect()[0].asDict()

original = json.loads(metadata_result[original_metadata_col])
deid = json.loads(metadata_result[deid_metadata_col])

In [0]:
original

{'00080008': {'tag': '00080008',
  'vr': 'CS',
  'value': ['DERIVED', 'SECONDARY']},
 '00080016': {'tag': '00080016',
  'vr': 'UI',
  'value': '1.2.840.10008.5.1.4.1.1.7.2'},
 '00080018': {'tag': '00080018',
  'vr': 'UI',
  'value': '1.2.826.0.1.3680043.8.498.39879364290725891944876183305437383239'},
 '00080020': {'tag': '00080020', 'vr': 'DA', 'value': '20260314'},
 '00080023': {'tag': '00080023', 'vr': 'DA', 'value': '20261005'},
 '00080030': {'tag': '00080030', 'vr': 'TM', 'value': '153812'},
 '00080033': {'tag': '00080033', 'vr': 'TM', 'value': '144130'},
 '00080050': {'tag': '00080050', 'vr': 'SH', 'value': 'ACC-20260314-007'},
 '00080060': {'tag': '00080060', 'vr': 'CS', 'value': 'OT'},
 '00080064': {'tag': '00080064', 'vr': 'CS', 'value': 'WSD'},
 '00080090': {'tag': '00080090', 'vr': 'PN', 'value': 'Nakamura^Kenji'},
 '0008103E': {'tag': '0008103E',
  'vr': 'LO',
  'value': 'Repeated single frame - test data'},
 '00082111': {'tag': '00082111',
  'vr': 'ST',
  'value': 'Source i

In [0]:
deid

{'00080008': {'tag': '00080008',
  'vr': 'CS',
  'value': ['DERIVED', 'SECONDARY']},
 '00080016': {'tag': '00080016',
  'vr': 'UI',
  'value': '1.2.840.10008.5.1.4.1.1.7.2'},
 '00080018': {'tag': '00080018',
  'vr': 'UI',
  'value': '2.25.211359326781749817404815063492775847523'},
 '00080020': {'tag': '00080020', 'vr': 'DA', 'value': '20260117'},
 '00080023': {'tag': '00080023', 'vr': 'DA', 'value': '20260823'},
 '00080030': {'tag': '00080030', 'vr': 'TM', 'value': '131800'},
 '00080033': {'tag': '00080033', 'vr': 'TM', 'value': '051118'},
 '00080050': {'tag': '00080050', 'vr': 'SH', 'value': ''},
 '00080060': {'tag': '00080060', 'vr': 'CS', 'value': 'OT'},
 '00080064': {'tag': '00080064', 'vr': 'CS', 'value': 'WSD'},
 '00080068': {'tag': '00080068', 'vr': 'CS', 'value': 'NONE'},
 '00080090': {'tag': '00080090', 'vr': 'PN', 'value': ''},
 '0008103E': {'tag': '0008103E',
  'vr': 'LO',
  'value': 'Repeated single frame - test data'},
 '00082111': {'tag': '00082111',
  'vr': 'ST',
  'valu

In [0]:
df = spark.read.format("binaryFile").load(dicom_root).withColumn("path", F.regexp_replace(F.col("path"), "dbfs:", ""))

print(f"Total Deid Files: {df.count()}")

Total Deid Files: 110


In [0]:
display_dicom(df=df, fields="content", limit=5, limit_frame=1, width=900, show_meta=False)